[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/cs377/blob/main/supplemental/Threshold_Pair.ipynb)

# Picking the Threshold: What Does a Mistake Cost?
**CS 377 · companion to 18_1_4 ROC Curves and Threshold Tuning · pair activity**

**Driver:** this notebook is yours. Your partner, the **Recorder**, has the **Picking the Threshold quiz in Canvas** open, reads each question aloud and types your agreed answers.

**How the two fit together.** The quiz is the guide; this notebook is where you do the work for the quiz questions tagged **💻 NOTEBOOK**. Each section below is named after its quiz question (**Q3** here is **Q3** in the quiz). Questions tagged **💬 NO NOTEBOOK** have no section here: just talk them through.

**For each notebook question:** the code is written for you. The Recorder reads the question aloud; you find the matching section, make the small change marked **✏️** and run the cell (Shift+Enter). Read the result aloud and agree with your partner on what it means. Hints are under each section: try without them first, and open Hint 1 before Hint 2.

## Setup (run this first)
**Part 1 of the quiz (Q1-Q2) has no notebook work.** Run this cell while the Recorder reads Part 1; loading the data and training the models takes a few seconds.

It rebuilds the 18_1_4 model (German Credit, the same split, XGBoost **without** `scale_pos_weight`) and makes what every section uses:
- `X_train`, `X_test`, `y_train`, `y_test`: 700 training and 300 test applicants; in `y`, 1 means the applicant defaulted and 0 means they repaid.
- `y_proba`: the model's predicted probability of default for each test applicant.
- `oof_proba`: out-of-fold probabilities for the 700 training applicants, computed with `cross_val_predict` as in 18_1_4.
- `make_model(weight)`: the same XGBoost model with `scale_pos_weight=weight`, not yet trained. `SPW` is the 2.33 weight from Beyond Accuracy.
- `THRESHOLDS`: the 100 thresholds from 0.01 to 0.99 that 18_1_4 tries.
- `cost_per_applicant(y_true, proba, threshold, cost_fp, cost_fn)`: the average cost per applicant when everyone at or above the threshold is flagged.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.metrics import roc_auc_score, roc_curve, recall_score, average_precision_score

pd.set_option('display.width', 140)

# German Credit, prepared exactly as in 18_1_4
df = fetch_openml(name='credit-g', version=1, as_frame=True).frame
y = (df['class'] == 'bad').astype(int)        # 1 = defaulted (bad credit), 0 = repaid (good credit)
X = df.drop(columns=['class'])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

SPW = (y_train == 0).sum() / (y_train == 1).sum()   # the Beyond Accuracy weight: 490 / 210, about 2.33


def make_model(weight):
    """The 18_1_1 XGBoost model, with scale_pos_weight set to `weight` (18_1_4 uses weight 1)."""
    return xgb.XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1, scale_pos_weight=weight,
                             eval_metric='logloss', random_state=42, enable_categorical=True, tree_method='hist')


def cost_per_applicant(y_true, proba, threshold, cost_fp, cost_fn):
    """Average cost per applicant when everyone with proba >= threshold is flagged (as in 18_1_4)."""
    pred = (np.asarray(proba) >= threshold).astype(int)
    y_true = np.asarray(y_true)
    fp = ((pred == 1) & (y_true == 0)).sum()
    fn = ((pred == 0) & (y_true == 1)).sum()
    return (fp * cost_fp + fn * cost_fn) / len(y_true)


THRESHOLDS = np.linspace(0.01, 0.99, 100)       # the thresholds 18_1_4 tries

model = make_model(1).fit(X_train, y_train)     # the 18_1_4 model: no scale_pos_weight
y_proba = model.predict_proba(X_test)[:, 1]     # P(default) for the 300 test applicants
oof_proba = cross_val_predict(make_model(1), X_train, y_train, cv=5,
                              method='predict_proba')[:, 1]   # out-of-fold P(default) for the 700 training applicants

print(f"training applicants: {len(y_train)}   test applicants: {len(y_test)}   test defaulters: {y_test.sum()}")

# Part 2 (Q3-Q6): The 18_1_4 Model

## Q3 · Out-of-fold probabilities
> **Quiz Q3:** The Q3 cell computes out-of-fold predictions for the 700 training applicants and their AUC, but it leaves out `method='predict_proba'`, so it gets 0/1 predictions instead of probabilities. Run it and note the AUC. Then add `method='predict_proba'` and `[:, 1]` as in 18_1_4 and run it again. Why is the AUC from 0/1 predictions lower? (Think about how many points the ROC curve can have.) Each out-of-fold probability was produced by which model, and had that model seen the applicant?

**✏️ Your change:** In the `cross_val_predict(...)` line, add `method='predict_proba'` and put `[:, 1]` after the closing parenthesis.

**Tell your Recorder:** the two AUCs, why 0/1 predictions give a lower AUC, and which model made each out-of-fold probability.

<details><summary>Hint 1</summary>

`predict_proba` returns two columns, one per class. `[:, 1]` keeps the probability of default.

</details>

<details><summary>Hint 2</summary>

`cross_val_predict(make_model(1), X_train, y_train, cv=5, method='predict_proba')[:, 1]`

</details>

In [ ]:
oof_q3 = cross_val_predict(make_model(1), X_train, y_train, cv=5)   # ✏️ probabilities, not 0/1

print('first five out-of-fold predictions:', np.round(oof_q3[:5], 3))
print(f'out-of-fold AUC: {roc_auc_score(y_train, oof_q3):.3f}')

## Q4 · Same AUC, different recall
> **Quiz Q4:** In Beyond Accuracy, `scale_pos_weight` ≈ 2.33 raised recall at the 0.5 threshold from 0.47 to 0.60. The Q4 cell compares two models on the test set, but right now both use weight 1. Change the second weight to `SPW` and run it. How much do the two AUCs differ? How much do the recalls at 0.5 differ? Use the mean P(default) column to explain how the weight can change recall at 0.5 so much while barely changing the AUC. (These AUCs are on the 300 test applicants, about 0.81; Q3 and 18_1_4 report the out-of-fold AUC on the 700 training applicants, about 0.76.)

**✏️ Your change:** Change the second weight in the line marked ✏️ from `1` to `SPW`, then run the cell.

**Tell your Recorder:** the two AUCs, the two recalls at 0.5, the two mean probabilities, then your explanation.

<details><summary>Hint 1</summary>

`SPW` is the 2.33 weight from Setup, the same one Beyond Accuracy used.

</details>

<details><summary>Hint 2</summary>

`weights = [1, SPW]`

</details>

In [ ]:
weights = [1, 1]      # ✏️ change the second weight to SPW (about 2.33)

rows = []
for w in weights:
    p = make_model(w).fit(X_train, y_train).predict_proba(X_test)[:, 1]
    rows.append({'model': f'weight {w:.2f}', 'test AUC': roc_auc_score(y_test, p),
                 'recall at 0.5': recall_score(y_test, p >= 0.5), 'mean P(default)': p.mean()})
pd.DataFrame(rows).set_index('model').round(3)

## Q5 · The cost ratio moves the threshold
> **Quiz Q5:** 18_1_4 assumed a missed default costs $5,000 and a false alarm $500 (10 to 1). The Q5 cell finds the threshold with the lowest cost on the out-of-fold training probabilities, then reports how that threshold does on the test set. Right now it tries only $5,000. Change `FN_COSTS` to `[5000, 1500, 500]` (10 to 1, 3 to 1 and 1 to 1) and run it. How does the chosen threshold move as a missed default gets cheaper, and what happens to recall? The table also shows the test AUC in every row: why can't changing the threshold change the AUC? Your manager asks, "What threshold should we use?" What do you need to ask them first?

**✏️ Your change:** Change `FN_COSTS = [5000]` to `FN_COSTS = [5000, 1500, 500]`, then run the cell.

**Tell your Recorder:** the three thresholds and recalls, why the AUC cannot change, then what you would ask the manager.

<details><summary>Hint 1</summary>

`FN_COSTS` is a list. Each value is the cost of one missed default; a false alarm always costs `COST_FP = 500`.

</details>

<details><summary>Hint 2</summary>

`FN_COSTS = [5000, 1500, 500]`

</details>

In [ ]:
COST_FP = 500
FN_COSTS = [5000]      # ✏️ try [5000, 1500, 500]

rows, curves = [], []
for cost_fn in FN_COSTS:
    oof_costs = [cost_per_applicant(y_train, oof_proba, t, COST_FP, cost_fn) for t in THRESHOLDS]
    best_t = THRESHOLDS[np.argmin(oof_costs)]        # chosen on training data only
    pred = y_proba >= best_t
    rows.append({'FN : FP cost': f'{cost_fn // COST_FP} to 1', 'threshold': best_t,
                 'test recall': recall_score(y_test, pred),
                 'test false alarms': int(((pred == 1) & (y_test == 0)).sum()),
                 'test AUC': roc_auc_score(y_test, y_proba)})
    curves.append(pd.DataFrame({'threshold': THRESHOLDS, 'cost per applicant': oof_costs,
                                'FN : FP cost': f'{cost_fn // COST_FP} to 1'}))

display(pd.DataFrame(rows).set_index('FN : FP cost').round(3))
fig, axes = plt.subplots(1, len(curves), figsize=(4.5 * len(curves), 3.5), squeeze=False)
for ax, c in zip(axes[0], curves):
    sns.lineplot(data=c, x='threshold', y='cost per applicant', ax=ax)
    low = c['cost per applicant'].idxmin()
    ax.scatter(c.loc[low, 'threshold'], c.loc[low, 'cost per applicant'], color='red', s=60, zorder=5)
    ax.set_title(f"{c['FN : FP cost'].iloc[0]}: lowest cost at {c.loc[low, 'threshold']:.2f}")
fig.suptitle('Cost per applicant on the out-of-fold training probabilities (each panel has its own scale)')
plt.tight_layout()
plt.show()

## Q6 · Which data picks the threshold?
> **Quiz Q6:** A classmate's version of the 18_1_4 search, in the Q6 cell, picks the cheapest threshold by searching on the test probabilities (10 to 1 costs). Run it and note the threshold and the test cost. Then change the search to use the out-of-fold training probabilities, as 18_1_4 does, and run it again. Which version reports the lower test cost? Why is that lower number not an honest estimate of what the bank will pay on new applicants? (Exam 2 Q8 and nested CV made the same point about choosing α.)

**✏️ Your change:** Change `search_y, search_proba = y_test, y_proba` to search on `y_train` and `oof_proba`.

**Tell your Recorder:** the two thresholds and test costs, which is lower, then why the test-chosen one is not honest.

<details><summary>Hint 1</summary>

`oof_proba` (from Setup) holds the out-of-fold probabilities for the 700 training applicants, so it goes with `y_train`.

</details>

<details><summary>Hint 2</summary>

`search_y, search_proba = y_train, oof_proba`

</details>

In [ ]:
COST_FP, COST_FN = 500, 5000
search_y, search_proba = y_test, y_proba        # ✏️ search on the out-of-fold training probabilities

costs = [cost_per_applicant(search_y, search_proba, t, COST_FP, COST_FN) for t in THRESHOLDS]
best_t = THRESHOLDS[np.argmin(costs)]
print(f'searched on {len(search_y)} applicants -> threshold {best_t:.2f}')
print(f'cost per applicant on the 300 test applicants: '
      f'${cost_per_applicant(y_test, y_proba, best_t, COST_FP, COST_FN):.2f}')

# Extra credit (Q7-Q8): Two More Views of the Same Model
Only if you have finished Q1-Q6.

## ⭐ Q7 · Precision-recall instead of ROC
> **Quiz Q7:** 18_1_4 says the precision-recall curve can be more informative than ROC on imbalanced data. In the Q7 cell, replace `np.nan` with the average precision of the test probabilities and run it. Compare it with the ROC AUC, and each with what random guessing would score (0.5 for ROC AUC, the positive rate for average precision). Which of the four confusion-matrix cells does the ROC curve use that the precision-recall curve ignores, and why does that matter more when defaults are rare?

**✏️ Your change:** Replace `np.nan` with `average_precision_score(y_test, y_proba)`.

**Tell your Recorder:** the two scores and their random baselines, the cell the PR curve ignores, and why it matters.

<details><summary>Hint 1</summary>

`average_precision_score` (imported in Setup) takes the true labels and the probabilities, like `roc_auc_score`.

</details>

<details><summary>Hint 2</summary>

`ap = average_precision_score(y_test, y_proba)`

</details>

In [ ]:
ap = np.nan      # ✏️ average precision of the test probabilities

print(f'ROC AUC:           {roc_auc_score(y_test, y_proba):.3f}   (random guessing: 0.500)')
print(f'average precision: {ap:.3f}   (random guessing: {y_test.mean():.3f})')

## ⭐ Q8 · What Youden's J assumes
> **Quiz Q8:** Youden's J picks the threshold where J = TPR − FPR is largest. In the Q8 cell, replace `np.nan` with that formula and run it. Which threshold does J choose, and which cost ratio in Q5 chose about the same one? 18_1_4 says J treats both kinds of error as equally costly, in the sense that it counts one percentage point of TPR the same as one point of FPR. The training set has 210 defaulters and 490 who repaid: how many applicants is one point of each worth, and so how many false alarms does J treat as costing the same as one missed default?

**✏️ Your change:** Replace `np.nan` with `tpr - fpr`.

**Tell your Recorder:** J's threshold, the matching Q5 ratio, then the per-applicant cost ratio J assumes and why.

<details><summary>Hint 1</summary>

`roc_curve` returns arrays `fpr` and `tpr`, one entry per threshold, so `tpr - fpr` is J for every threshold at once.

</details>

<details><summary>Hint 2</summary>

`j = tpr - fpr`

</details>

In [ ]:
fpr, tpr, roc_thresholds = roc_curve(y_train, oof_proba)

j = np.nan      # ✏️ Youden's J for every threshold

best = np.argmax(j)
print(f"Youden's J threshold: {roc_thresholds[best]:.3f}   (TPR {tpr[best]:.3f}, FPR {fpr[best]:.3f})")
print(f'training defaulters: {(y_train == 1).sum()}   training repaid: {(y_train == 0).sum()}')